# 14 — Ensembling and Stacking (Lesson)

## Problem Definition
Combine diverse models to reduce variance and improve leaderboard stability.

## Required Prior Knowledge
- Notebook 12 OOF/CV discipline.
- Notebook 13 leakage-safe diagnostics.

## New Concepts Introduced
- Bagging variance reduction with correlation terms.
- Ensemble covariance formula.
- Stacking as second-level ERM.
- OOF stacking implementation.
- Blending vs stacking empirical comparison.

## Formal Definition
For ensemble
$$
\hat{y}_{ens}=\sum_{m=1}^M w_m\hat{y}^{(m)},\quad \sum_m w_m=1,
$$
variance is
$$
\mathrm{Var}(\hat{y}_{ens})=\sum_{m=1}^M\sum_{\ell=1}^M w_m w_\ell\,\mathrm{Cov}(\hat{y}^{(m)},\hat{y}^{(\ell)}).
$$

Equal-variance equal-correlation case ($\sigma^2,\rho$):
$$
\mathrm{Var}(\bar{y})=\sigma^2\left(\rho+\frac{1-\rho}{M}\right).
$$

Stacking objective:
$$
\theta^*=\arg\min_\theta\frac{1}{n}\sum_{i=1}^n
L\left(y_i,g_\theta\left(\hat{y}^{(1)}_{i,\mathrm{OOF}},\dots,\hat{y}^{(M)}_{i,\mathrm{OOF}}\right)\right).
$$

## Variables and Assumptions
- Scope: Ensembling and Stacking targets the objective `Combine diverse models to reduce variance and improve leaderboard stability.`.
- Data are indexed by $i\in\{1,\dots,n\}$ with features $x_i\in\mathbb{R}^d$ and target $y_i$.
- Model parameters are represented by $\theta$, and predictions are $\hat{y}_i=f_\theta(x_i)$.
- Any preprocessing statistics are fit on training partitions only and reused on validation/test partitions.
- Split protocol (KFold/Group/TimeSeries) must match the data-generating assumptions for IID, grouped, or temporal samples.
- Reported metrics are empirical estimates with finite-sample variance; interpretation must include uncertainty.

## Symbol-by-Symbol Explanation
| Symbol | Meaning |
|---|---|
| $x_i$ | feature vector for sample $i$ |
| $y_i$ | target for sample $i$ |
| $f_\theta$ | model parameterized by $\theta$ |
| $L(\cdot,\cdot)$ | per-sample loss function |
| $n$ | number of samples |
| $d$ | raw feature dimension |
| $p$ | transformed feature dimension / polynomial degree context |
| $K$ | number of folds / partitions |
| $V_k$ | validation index set for fold $k$ |
| $\lambda,\alpha,\tau$ | regularization/smoothing/confidence hyperparameters |

## Zero-Skip Derivation
1. Write ensemble prediction as weighted sum.
2. Apply variance bilinearity:
   $$
   \mathrm{Var}\left(\sum_m w_m Z_m\right)=\sum_m\sum_\ell w_m w_\ell\mathrm{Cov}(Z_m,Z_\ell).
   $$
3. Substitute $Z_m=\hat{y}^{(m)}$.
4. In equal-correlation case:
   - diagonal terms: $M\cdot (1/M^2)\sigma^2 = \sigma^2/M$
   - off-diagonal terms: $M(M-1)\cdot (1/M^2)\rho\sigma^2 = \rho\sigma^2(1-1/M)$
5. Sum both terms to get
   $$
   \sigma^2\left(\rho+\frac{1-\rho}{M}\right).
   $$

## Explicit Logical Transitions
0. Context anchor: this notebook focuses on `Ensembling and Stacking` and objective `Combine diverse models to reduce variance and improve leaderboard stability.`.
1. Start from the formal objective and identify estimators/transformations introduced in this notebook.
2. Map each estimator term to computable quantities under train/validation split constraints.
3. Show why each derivation step is valid (algebraic identity, estimator definition, or probabilistic assumption).
4. Convert the derivation into an implementation protocol with explicit leakage controls.
5. Validate with synthetic and real-data experiments, then interpret failure conditions.

## Intuition
Ensembles only help when members are not perfectly correlated; OOF stacking learns data-driven combination weights while preserving validation integrity.

## Mapping from Math to Implementation
- `bagging_variance_formula` computes theoretical reduction.
- `ensemble_variance_from_covariance` evaluates empirical covariance effects.
- `oof_stacking` enforces leakage-safe meta-feature creation.
- `blend_predictions` implements static weighted averaging.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn

from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

from feature_utils import (
    set_global_seed,
    standardize_train_valid,
    monotone_log1p,
    one_hot_basis,
    polynomial_basis,
    add_interaction_columns,
    target_encode_oof,
    conditional_expectation_feature,
)
from cv_utils import (
    empirical_risk,
    make_splitter,
    oof_cv_predictions,
    cv_bias_variance_decomposition,
    leakage_inflation,
    simulate_public_private_variance,
)
from ensemble_utils import (
    blend_predictions,
    bagging_variance_formula,
    ensemble_variance_from_covariance,
    oof_stacking,
    stacking_predict,
)
from experiment_logger import ExperimentLogger, ExperimentRecord

SEED = 42
set_global_seed(SEED)

MODULE_DIR = Path('.').resolve()

## Synthetic Experiment

In [ ]:
rng = np.random.default_rng(SEED)
n = 5000
signal = rng.normal(size=n)
eps = rng.normal(size=(n, 3))

# Correlated predictors around shared signal.
pred1 = signal + 0.9 * eps[:, 0]
pred2 = signal + 0.9 * (0.7 * eps[:, 0] + 0.3 * eps[:, 1])
pred3 = signal + 0.9 * (0.5 * eps[:, 0] + 0.5 * eps[:, 2])
pred_matrix = np.column_stack([pred1, pred2, pred3])
cov = np.cov(pred_matrix, rowvar=False)

w = np.array([1/3, 1/3, 1/3])
var_emp = ensemble_variance_from_covariance(cov, w)
print({"empirical_weighted_variance": var_emp, "cov_matrix": cov})
print({"theory_example": bagging_variance_formula(base_variance=1.0, pairwise_corr=0.6, n_models=3)})

## Real Dataset Experiment

In [ ]:
bc = load_breast_cancer(as_frame=True)
X = bc.data.values
y = bc.target.values
splitter = make_splitter("stratified", n_splits=5, seed=SEED)

base_models = [
    ("lr", LogisticRegression(max_iter=2000)),
    ("rf", RandomForestClassifier(n_estimators=250, random_state=SEED)),
    ("hgb", HistGradientBoostingClassifier(random_state=SEED)),
]
meta = LogisticRegression(max_iter=2000)

stack_bundle = oof_stacking(base_models, meta, X, y, splitter, task="classification")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)

# Refit base models on train for direct blend baseline.
trained = []
for name, est in base_models:
    fit_est = est.fit(X_train, y_train)
    trained.append((name, fit_est))
preds = np.column_stack([m.predict_proba(X_test)[:, 1] for _, m in trained])
blend_pred = blend_predictions(preds, [0.25, 0.4, 0.35])

stack_pred = stacking_predict(stack_bundle, X_test, task="classification")
print({
    "blend_auc": roc_auc_score(y_test, blend_pred),
    "stack_auc": roc_auc_score(y_test, stack_pred),
})

## Diagnostic Analysis

In [ ]:
pred_corr = np.corrcoef(preds, rowvar=False)
print("base prediction correlation matrix:\n", pred_corr)
plt.figure(figsize=(4, 3))
plt.imshow(pred_corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.title("Prediction correlation")
plt.tight_layout()
plt.show()

## Failure Analysis

In [ ]:
# Failure case: training meta-model on in-sample base predictions (leaky stacking).
leaky_meta_X = np.column_stack([m.predict_proba(X_train)[:, 1] for _, m in trained])
leaky_meta = LogisticRegression(max_iter=2000).fit(leaky_meta_X, y_train)
test_meta_X = np.column_stack([m.predict_proba(X_test)[:, 1] for _, m in trained])
leaky_pred = leaky_meta.predict_proba(test_meta_X)[:, 1]
print({"leaky_stack_auc": roc_auc_score(y_test, leaky_pred), "proper_stack_auc": roc_auc_score(y_test, stack_pred)})

## Exercise Ladder (basic → advanced → research-level)
1. Derive optimal 2-model blend weight under quadratic loss and known covariance matrix.
2. Prove blending is a restricted linear stacking case.
3. Add XGBoost and LightGBM base learners and quantify diversity gain.
4. Evaluate stack robustness under fold perturbation.

## Summary of Mathematical Insights
Stacking is controlled meta-learning over OOF predictions; variance reduction depends on diversity, not model count alone.